# PPE Detection: training on Colab

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

What it does:
1. Downloads the 4 Kaggle datasets and merges them into one 8-class YOLO dataset (`prepare_data.py`).
2. Trains a **baseline** YOLOv8s and a **balanced** YOLOv8s that sees rare classes (goggles, no-goggles, no-mask) more often through repeat-factor sampling.
3. Optionally trains **YOLOv5s** on the same split for a model comparison.
4. Evaluates every run on the held-out test split and saves weights, metrics and plots to Google Drive.

Everything is saved under `MyDrive/ppe-detection`. If Colab disconnects, run all again: finished runs are skipped and unfinished ones resume from their last epoch.

In [ ]:
# Settings
EPOCHS = 25          # about 1.5 h per run on a free T4; raise if you have time
MODEL = "yolov8s.pt"
RUN_YOLOV5 = False   # set True for the extra YOLOv5 vs YOLOv8 comparison run (another ~1.5 h)
REPO = "https://github.com/NishaadG/ppe-detection.git"

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/ppe-detection'
import os
os.makedirs(WORK, exist_ok=True)

In [ ]:
%cd /content
!rm -rf ppe-detection && git clone -q {REPO}
%cd /content/ppe-detection
!pip -q install "ultralytics>=8.3.0" kaggle

In [ ]:
# Kaggle API key: put kaggle.json in MyDrive/ppe-detection once, or upload it when asked
import shutil
os.makedirs('/root/.kaggle', exist_ok=True)
if os.path.exists(f'{WORK}/kaggle.json'):
    shutil.copy(f'{WORK}/kaggle.json', '/root/.kaggle/kaggle.json')
else:
    from google.colab import files
    up = files.upload()
    shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
    shutil.copy('kaggle.json', f'{WORK}/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
DATASETS = {
    "helmet": "andrewmvd/hard-hat-detection",
    "mask": "andrewmvd/face-mask-detection",
    "vest": "adilshamim8/safety-vests-detection-dataset",
    "goggles": "ketakichalke/ppe-kit-detection-construction-site-workers",
}
for name, slug in DATASETS.items():
    if not os.path.exists(f'/content/raw/{name}'):
        !kaggle datasets download -d {slug} -p /content/raw/{name} --unzip -q
!python prepare_data.py --raw /content/raw --out /content/data --seed 42
!cp /content/data/stats.json /content/data/class_distribution.png {WORK}/
from IPython.display import Image, display
display(Image('/content/data/class_distribution.png'))

Check the printed class names of the `vest` and `goggles` source datasets against the mappings at the top of `prepare_data.py` (`YOLO_MAP`). They match the original project, but confirm once.

In [ ]:
import csv
import torch
from ultralytics import YOLO

def epochs_done(run_dir):
    f = f'{run_dir}/results.csv'
    if not os.path.exists(f):
        return 0
    with open(f) as fh:
        return max(0, sum(1 for _ in fh) - 1)

def finished(run_dir):
    # Ultralytics sets epoch=-1 in last.pt when training ends, including early stops (patience),
    # so an early-stopped run is not mistaken for an interrupted one and retrained from scratch.
    last = f'{run_dir}/weights/last.pt'
    return os.path.exists(last) and torch.load(last, map_location='cpu', weights_only=False).get('epoch', -1) == -1

def train(name, data, model=MODEL, epochs=EPOCHS):
    run_dir = f'{WORK}/runs/{name}'
    done = epochs_done(run_dir)
    if finished(run_dir) and os.path.exists(f'{run_dir}/weights/best.pt'):
        print(f'{name}: already finished ({done} epochs)')
        return
    last = f'{run_dir}/weights/last.pt'
    if os.path.exists(last):
        print(f'{name}: resuming after epoch {done}')
        try:
            YOLO(last).train(resume=True)
            return
        except Exception as e:
            print('resume failed, starting again:', e)
    YOLO(model).train(data=data, epochs=epochs, imgsz=640, batch=16, seed=42, patience=10,
                      workers=2, project=f'{WORK}/runs', name=name, exist_ok=True, plots=True)

In [ ]:
train('baseline', '/content/data/ppe.yaml')

In [ ]:
train('balanced', '/content/data/ppe_balanced.yaml')

In [ ]:
if RUN_YOLOV5:
    train('yolov5s', '/content/data/ppe.yaml', model='yolov5su.pt')

In [ ]:
runs = [f'{WORK}/runs/{n}' for n in ('baseline', 'balanced', 'yolov5s') if os.path.exists(f'{WORK}/runs/{n}/weights/best.pt')]
runs_arg = ' '.join(runs)
!python evaluate.py --data /content/data/ppe.yaml --runs {runs_arg} --out {WORK}/results

In [ ]:
best = open(f'{WORK}/results/best_run.txt').read().strip()
os.makedirs(f'{WORK}/weights', exist_ok=True)
shutil.copy(f'{WORK}/runs/{best}/weights/best.pt', f'{WORK}/weights/best.pt')
print('Best run:', best, '-> copied to', f'{WORK}/weights/best.pt')
for f in ('confusion_matrix_normalized.png', 'PR_curve.png'):
    p = f'{WORK}/results/{best}_test/{f}'
    if os.path.exists(p):
        display(Image(p, width=700))

In [ ]:
# A few test-set predictions, saved for the README
import glob, random
random.seed(0)
imgs = random.sample(sorted(glob.glob('/content/data/images/test/*')), 6)
model = YOLO(f'{WORK}/weights/best.pt')
os.makedirs(f'{WORK}/samples', exist_ok=True)
for i, p in enumerate(imgs):
    r = model.predict(p, conf=0.35, verbose=False)[0]
    out = f'{WORK}/samples/sample_{i}.jpg'
    r.save(filename=out)
    display(Image(out, width=420))

## Done
Send these back (they are all in `MyDrive/ppe-detection`):
- `results/comparison.md` (the table for the README and resume)
- `stats.json`
- `weights/best.pt` (goes into the Hugging Face Space)
- `samples/` and `results/<best>_test/` plots (for the README)